# **I. Environment and Path Setup**

## A. Dependency Verification & Installation

In [1]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "mlxtend": "mlxtend",
}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing {pip_name} into active kernel ({sys.executable})...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("All required packages are available in the active kernel.")

All required packages are available in the active kernel.


Checks whether pandas, numpy, matplotlib, seaborn, scikit-learn, and lmxtend are already installed in the active Jupyter Kernet and installs any missing packages via pip so the notebook runs reliably in both VS Code and Google Colab. 

## B. Runtime Environment Detection & Repository Setup

In [2]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
GITHUB_USER = "ShaunCipher"
REPO_NAME = "Jusgado_data-mining-capstone"
BRANCH = "main"

if IN_COLAB:
    repo_dir = Path(f"/content/{REPO_NAME}")
    if not repo_dir.exists():
        print(f"Cloning {GITHUB_USER}/{REPO_NAME} ({BRANCH}) for Google Colab...")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "-b",
                BRANCH,
                f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
                str(repo_dir),
            ],
            check=True,
        )
    PROJECT_ROOT = repo_dir
else:
    cwd = Path.cwd()
    if (cwd / "datasets").exists():
        PROJECT_ROOT = cwd
    elif (cwd.parent / "datasets").exists():
        PROJECT_ROOT = cwd.parent
    else:
        PROJECT_ROOT = cwd

print("Runtime Environment:", "Google Colab" if IN_COLAB else "Local VS Code")
print("Python Executable:", sys.executable)
print("Project Root:", PROJECT_ROOT.resolve())

Runtime Environment: Local VS Code
Python Executable: c:\Users\Jusgado\AppData\Local\Python\pythoncore-3.14-64\python.exe
Project Root: C:\Users\Jusgado\OneDrive\Documents\Github\Jusgado_data-mining-capstone


Detects whether the notebook is running in Google Colab or locally in VS Code, clones ShaunCipher/Jusgado_data-mining-capstone into /content when executed in Colab, and resolves PROJECT_ROOT consistently across local and cloud runs.

## C. Dataset Directory & File Inventory Check

In [3]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "datasets"
assert DATA_DIR.exists(), f"Datasets folder not found at: {DATA_DIR.resolve()}"

discovered_items = sorted(
    [p.relative_to(DATA_DIR) for p in DATA_DIR.rglob("*") if p.is_file()]
)

print("Resolved DATA_DIR:", DATA_DIR.resolve())
print(f"Total files discovered under datasets/: {len(discovered_items)}")
for item in discovered_items:
    print(" -", item)

Resolved DATA_DIR: C:\Users\Jusgado\OneDrive\Documents\Github\Jusgado_data-mining-capstone\datasets
Total files discovered under datasets/: 76
 - menu_items\menu_items.csv
 - payment_methods\payment_methods.csv
 - stores\stores.csv
 - transaction_items\transaction_items_202307.csv
 - transaction_items\transaction_items_202308.csv
 - transaction_items\transaction_items_202309.csv
 - transaction_items\transaction_items_202310.csv
 - transaction_items\transaction_items_202311.csv
 - transaction_items\transaction_items_202312.csv
 - transaction_items\transaction_items_202401.csv
 - transaction_items\transaction_items_202402.csv
 - transaction_items\transaction_items_202403.csv
 - transaction_items\transaction_items_202404.csv
 - transaction_items\transaction_items_202405.csv
 - transaction_items\transaction_items_202406.csv
 - transaction_items\transaction_items_202407.csv
 - transaction_items\transaction_items_202408.csv
 - transaction_items\transaction_items_202409.csv
 - transaction_ite

Verifies that the datasets/ directory exists under PROJECT_ROOT and lists every file found inside it in order to confirm the raw dataset files are in place before loading them into SQLite.

Brief Description of Datasets: 
*  menu_items.csv - Contains the items available at the menu
*  payment_methods.csv - Contains the available payment method 
*  stores.csv - Contains participating stores address
*  transaction_items_yyyymm.csv - Contains the transactions information regarding items from July of 2023 to March of 2025
*  transaction_yyyymm.csv - Contains the transactions information regarding customers from July of 2023 to June of 2025
*  users_yyyymm.csv - Contains the customer information July of 2023 to June of 2025
*  vouchers.csv - Contains available vouchers for transactions

## D. Load CSV Datasets into SQLite & Summarize Schemas

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
csv_files = sorted(DATA_DIR.rglob("*.csv"))
assert len(csv_files) > 0, f"No CSV files found under {DATA_DIR.resolve()}"

summary_rows = []
loaded_tables = {}

for csv_path in csv_files:
    table_name = csv_path.stem.lower().replace("-", "_").replace(" ", "_")
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    loaded_tables[table_name] = df

    summary_rows.append(
        {
            "table_name": table_name,
            "file": str(csv_path.relative_to(DATA_DIR)),
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(df.columns),
        }
    )

schema_summary_df = pd.DataFrame(summary_rows)
display(schema_summary_df)

,table_name,file,rows,cols,columns
0,menu_items,menu_items\menu_items.csv,8,7,"item_id, item_name, category, price, is_season..."
1,payment_methods,payment_methods\payment_methods.csv,5,3,"method_id, method_name, category"
2,stores,stores\stores.csv,10,8,"store_id, store_name, street, postal_code, cit..."
3,transaction_items_202307,transaction_items\transaction_items_202307.csv,1241261,6,"transaction_id, item_id, quantity, unit_price,..."
4,transaction_items_202308,transaction_items\transaction_items_202308.csv,1238735,6,"transaction_id, item_id, quantity, unit_price,..."
...,...,...,...,...,...
71,users_202503,users\users_202503.csv,182902,4,"user_id, gender, birthdate, registered_at"
72,users_202504,users\users_202504.csv,178617,4,"user_id, gender, birthdate, registered_at"
73,users_202505,users\users_202505.csv,184015,4,"user_id, gender, birthdate, registered_at"
74,users_202506,users\users_202506.csv,178713,4,"user_id, gender, birthdate, registered_at"


Reads all discovered CSV files into Pandas DataFrames, loads each table into an in-memory SQLite database for SQL querying, stores the DataFrames in loaded_tables, and displays a summary table of row counts, column counts, and column names.

# **II. Data Understanding**

## A. Table Schema, Null Count & Sample Rows Inspection

In [11]:
import pandas as pd


def inspect_subfolder(subfolder_name: str, preview_rows: int = 5):
  """Inspect all CSV files inside a subfolder under DATA_DIR with full details."""
  sub_path = DATA_DIR / subfolder_name
  assert sub_path.exists(), f"Subfolder not found: {sub_path.resolve()}"

  csv_files = sorted(sub_path.rglob("*.csv"))
  assert len(csv_files) > 0, f"No CSV files found under {sub_path.resolve()}"

  print(f"Subfolder: `{subfolder_name}` ({len(csv_files)} CSV file(s))")

  overview_rows = []
  dfs_by_stem = {}

  for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs_by_stem[csv_file.stem] = df
    overview_rows.append({
        "table": csv_file.stem,
        "rows": len(df),
        "cols": len(df.columns),
        "total_nulls": int(df.isna().sum().sum()),
        "columns": ", ".join(df.columns),
    })

  display(pd.DataFrame(overview_rows))

  for csv_file in csv_files:
    df = dfs_by_stem[csv_file.stem]
    print(f"Table: `{csv_file.stem}` ({len(df):,} rows)")
    col_info = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "nulls": df.isna().sum().values,
        "null_%": (df.isna().mean() * 100).round(2).values,
        "unique": df.nunique(dropna=True).values,
    })
    display(col_info)
    if len(df) > 0:
      display(df.head(preview_rows))

## B. Displaying the Results

Note: Please open the sub topics one at a time since there are too many tables

### i. `menu_items`
Inspects the product catalog table, item categories, pricing, and seasonal availability fields.

In [12]:
inspect_subfolder("menu_items", preview_rows=5)

Subfolder: `menu_items` (1 CSV file(s))


,table,rows,cols,total_nulls,columns
0,menu_items,8,7,16,"item_id, item_name, category, price, is_season..."


Table: `menu_items` (8 rows)


,column,dtype,nulls,null_%,unique
0,item_id,int64,0,0.0,8
1,item_name,str,0,0.0,8
2,category,str,0,0.0,2
3,price,float64,0,0.0,6
4,is_seasonal,bool,0,0.0,1
5,available_from,float64,8,100.0,0
6,available_to,float64,8,100.0,0


,item_id,item_name,category,price,is_seasonal,available_from,available_to
0,1,Espresso,coffee,6.0,False,NaN,NaN
1,2,Americano,coffee,7.0,False,NaN,NaN
2,3,Latte,coffee,8.0,False,NaN,NaN
3,4,Cappuccino,coffee,8.0,False,NaN,NaN
4,5,Flat White,coffee,9.0,False,NaN,NaN


### ii. `payment_methods`
Inspects available payment methods and payment categories used across transactions.

In [13]:
inspect_subfolder("payment_methods", preview_rows=5)

Subfolder: `payment_methods` (1 CSV file(s))


,table,rows,cols,total_nulls,columns
0,payment_methods,5,3,0,"method_id, method_name, category"


Table: `payment_methods` (5 rows)


,column,dtype,nulls,null_%,unique
0,method_id,int64,0,0.0,5
1,method_name,str,0,0.0,5
2,category,str,0,0.0,3


,method_id,method_name,category
0,1,cash,cash
1,2,credit_card,card
2,3,debit_card,card
3,4,tng,ewallet
4,5,grabpay,ewallet


### iii. `stores`
Inspects store locations, addresses, geographic coordinates, and branch identifiers.

In [18]:
inspect_subfolder("stores", preview_rows=11)

Subfolder: `stores` (1 CSV file(s))


,table,rows,cols,total_nulls,columns
0,stores,10,8,0,"store_id, store_name, street, postal_code, cit..."


Table: `stores` (10 rows)


,column,dtype,nulls,null_%,unique
0,store_id,int64,0,0.0,10
1,store_name,str,0,0.0,10
2,street,str,0,0.0,10
3,postal_code,int64,0,0.0,10
4,city,str,0,0.0,10
5,state,str,0,0.0,3
6,latitude,float64,0,0.0,10
7,longitude,float64,0,0.0,10


,store_id,store_name,street,postal_code,city,state,latitude,longitude
0,1,G Coffee @ USJ 89q,Jalan Dewan Bahasa 5/9,50998,USJ 89q,Kuala Lumpur,3.117134,101.615027
1,2,G Coffee @ Kondominium Putra,Jln Yew 6X,63826,Kondominium Putra,Selangor Darul Ehsan,2.959571,101.517720
2,3,G Coffee @ USJ 57W,Jalan Bukit Petaling 5/16C,62094,USJ 57W,Putrajaya,2.951038,101.663698
3,4,G Coffee @ Kampung Changkat,Jln 6/6A,62941,Kampung Changkat,Putrajaya,2.914594,101.704486
4,5,G Coffee @ Seksyen 21,Jalan Anson 4k,62595,Seksyen 21,Putrajaya,2.937599,101.698478
5,6,G Coffee @ Alam Tun Hussein Onn,Jln Pasar Besar 63s,63518,Alam Tun Hussein Onn,Selangor Darul Ehsan,3.279175,101.784923
6,7,G Coffee @ Damansara Saujana,Jln 8/74,65438,Damansara Saujana,Selangor Darul Ehsan,3.220810,101.584590
7,8,G Coffee @ Bandar Seri Mulia,Jalan Wisma Putra,58621,Bandar Seri Mulia,Kuala Lumpur,3.140674,101.706562
8,9,G Coffee @ PJS8,Jalan 7/3o,62418,PJS8,Putrajaya,2.952444,101.702623
9,10,G Coffee @ Taman Damansara,Jln 2,67102,Taman Damansara,Selangor Darul Ehsan,3.497178,101.595271


### iv. `transaction_items`
Inspects the monthly-partitioned line-item tables (`2023-07` to `2025-06`), covering item quantities, unit prices, subtotals, and timestamps.

In [19]:
inspect_subfolder("transaction_items", preview_rows=5)

Subfolder: `transaction_items` (24 CSV file(s))


,table,rows,cols,total_nulls,columns
0,transaction_items_202307,1241261,6,0,"transaction_id, item_id, quantity, unit_price,..."
1,transaction_items_202308,1238735,6,0,"transaction_id, item_id, quantity, unit_price,..."
2,transaction_items_202309,1200926,6,0,"transaction_id, item_id, quantity, unit_price,..."
3,transaction_items_202310,1238345,6,0,"transaction_id, item_id, quantity, unit_price,..."
4,transaction_items_202311,1201734,6,0,"transaction_id, item_id, quantity, unit_price,..."
5,transaction_items_202312,1240643,6,0,"transaction_id, item_id, quantity, unit_price,..."
6,transaction_items_202401,1240381,6,0,"transaction_id, item_id, quantity, unit_price,..."
7,transaction_items_202402,1159840,6,0,"transaction_id, item_id, quantity, unit_price,..."
8,transaction_items_202403,1240277,6,0,"transaction_id, item_id, quantity, unit_price,..."
9,transaction_items_202404,1199817,6,0,"transaction_id, item_id, quantity, unit_price,..."


Table: `transaction_items_202307` (1,241,261 rows)


,column,dtype,nulls,null_%,unique
0,transaction_id,str,0,0.0,620262
1,item_id,int64,0,0.0,8
2,quantity,int64,0,0.0,3
3,unit_price,float64,0,0.0,6
4,subtotal,float64,0,0.0,17
5,created_at,str,0,0.0,620262


,transaction_id,item_id,quantity,unit_price,subtotal,created_at
0,2ae6d188-76c2-4095-b861-ab97d3cd9312,6,3,9.5,28.5,2023-07-01 07:00:00
1,2ae6d188-76c2-4095-b861-ab97d3cd9312,6,1,9.5,9.5,2023-07-01 07:00:00
2,7d0a474d-62f4-442a-96b6-a5df2bda8832,5,3,9.0,27.0,2023-07-01 07:00:02
3,7d0a474d-62f4-442a-96b6-a5df2bda8832,1,1,6.0,6.0,2023-07-01 07:00:02
4,85f86fef-fddb-4eef-9dc3-1444553e6108,7,3,9.0,27.0,2023-07-01 07:00:04


Table: `transaction_items_202308` (1,238,735 rows)


... (64 outputs left out)

,transaction_id,item_id,quantity,unit_price,subtotal,created_at
0,2e88b589-f813-49ee-a84c-bde3c7990217,1,2,6.0,12.0,2025-05-01 16:17:29
1,b8a05324-c892-4e1f-a4b8-c78ec3884847,8,1,10.0,10.0,2025-05-01 10:51:41
2,b8a05324-c892-4e1f-a4b8-c78ec3884847,3,2,8.0,16.0,2025-05-01 10:51:41
3,06540f0c-b414-46ab-912d-714c35514e6c,6,2,9.5,19.0,2025-05-01 10:51:42
4,06540f0c-b414-46ab-912d-714c35514e6c,7,2,9.0,18.0,2025-05-01 10:51:42


Table: `transaction_items_202506` (1,199,052 rows)


,column,dtype,nulls,null_%,unique
0,transaction_id,str,0,0.0,599566
1,item_id,int64,0,0.0,8
2,quantity,int64,0,0.0,3
3,unit_price,float64,0,0.0,6
4,subtotal,float64,0,0.0,17
5,created_at,str,0,0.0,599566


,transaction_id,item_id,quantity,unit_price,subtotal,created_at
0,9aa9ee70-790b-40bb-9fd9-cada0afa7b0b,8,2,10.0,20.0,2025-06-01 11:53:34
1,da962c03-9e09-490a-be13-11bbe48e15ab,4,2,8.0,16.0,2025-06-01 08:42:45
2,867b79ec-0545-47b0-b798-ef14d41ee078,2,3,7.0,21.0,2025-06-01 08:42:46
3,867b79ec-0545-47b0-b798-ef14d41ee078,4,1,8.0,8.0,2025-06-01 08:42:46
4,867b79ec-0545-47b0-b798-ef14d41ee078,6,2,9.5,19.0,2025-06-01 08:42:46


inspect_subfolder("transactions", preview_rows=5)

transactions without vouchers are marked and counted as null since this process only involves reading the raw file without modifications yet.

In [20]:
inspect_subfolder("transactions", preview_rows=5)

Subfolder: `transactions` (24 CSV file(s))


,table,rows,cols,total_nulls,columns
0,transactions_202307,620262,9,1196363,"transaction_id, store_id, payment_method_id, v..."
1,transactions_202308,619980,9,1180855,"transaction_id, store_id, payment_method_id, v..."
2,transactions_202309,600643,9,1126137,"transaction_id, store_id, payment_method_id, v..."
3,transactions_202310,619396,9,1187959,"transaction_id, store_id, payment_method_id, v..."
4,transactions_202311,600545,9,1144611,"transaction_id, store_id, payment_method_id, v..."
5,transactions_202312,620777,9,1218008,"transaction_id, store_id, payment_method_id, v..."
6,transactions_202401,620358,9,1202166,"transaction_id, store_id, payment_method_id, v..."
7,transactions_202402,580012,9,1103156,"transaction_id, store_id, payment_method_id, v..."
8,transactions_202403,620481,9,1147192,"transaction_id, store_id, payment_method_id, v..."
9,transactions_202404,599847,9,1063368,"transaction_id, store_id, payment_method_id, v..."


Table: `transactions_202307` (620,262 rows)


,column,dtype,nulls,null_%,unique
0,transaction_id,str,0,0.00,620262
1,store_id,int64,0,0.00,10
2,payment_method_id,int64,0,0.00,5
3,voucher_id,float64,578009,93.19,1
4,user_id,float64,618354,99.69,580
5,original_amount,float64,0,0.00,146
6,discount_applied,float64,0,0.00,147
7,final_amount,float64,0,0.00,291
8,created_at,str,0,0.00,620262


,transaction_id,store_id,payment_method_id,voucher_id,user_id,original_amount,discount_applied,final_amount,created_at
0,2ae6d188-76c2-4095-b861-ab97d3cd9312,4,5,NaN,NaN,38.0,0.0,38.0,2023-07-01 07:00:00
1,7d0a474d-62f4-442a-96b6-a5df2bda8832,7,1,NaN,NaN,33.0,0.0,33.0,2023-07-01 07:00:02
2,85f86fef-fddb-4eef-9dc3-1444553e6108,1,5,NaN,NaN,27.0,0.0,27.0,2023-07-01 07:00:04
3,4c41d179-f809-4d5a-a5d7-acb25ae1fe98,5,2,NaN,NaN,45.5,0.0,45.5,2023-07-01 07:00:21
4,51e44c8e-4812-4a15-a9f9-9a46b62424d6,8,5,NaN,NaN,27.0,0.0,27.0,2023-07-01 07:00:33


Table: `transactions_202308` (619,980 rows)


... (64 outputs left out)

,transaction_id,store_id,payment_method_id,voucher_id,user_id,original_amount,discount_applied,final_amount,created_at
0,e75220ac-6b77-484f-9a19-d6d38785029b,1,4,16.0,92761.0,9.0,4.5,4.5,2025-05-01 10:13:52
1,37c46fcb-ab92-47f8-a25a-8987b147e939,8,5,16.0,1834540.0,48.0,24.0,24.0,2025-05-01 10:13:53
2,1e36c7bb-12bd-4292-9766-3fadaf719618,9,4,NaN,111159.0,22.0,0.0,22.0,2025-05-01 10:13:54
3,a56ddf4c-9039-4cd7-a958-9b758e566574,4,1,16.0,1834541.0,55.0,27.5,27.5,2025-05-01 10:13:55
4,1864ef2b-0538-4e16-9ca1-810091050e54,3,2,16.0,792361.0,18.0,9.0,9.0,2025-05-01 10:13:56


Table: `transactions_202506` (599,566 rows)


,column,dtype,nulls,null_%,unique
0,transaction_id,str,0,0.00,599566
1,store_id,int64,0,0.00,10
2,payment_method_id,int64,0,0.00,5
3,voucher_id,float64,557383,92.96,1
4,user_id,float64,6174,1.03,539376
5,original_amount,float64,0,0.00,146
6,discount_applied,float64,0,0.00,146
7,final_amount,float64,0,0.00,288
8,created_at,str,0,0.00,599566


,transaction_id,store_id,payment_method_id,voucher_id,user_id,original_amount,discount_applied,final_amount,created_at
0,cf97cf60-a84d-4072-9569-8fa2aaa1584f,10,3,NaN,2020374.0,20.0,0.0,20.0,2025-06-01 12:24:10
1,f827c197-bbfa-472b-952d-9909080f2056,1,1,NaN,2018692.0,68.0,0.0,68.0,2025-06-01 10:24:57
2,b9d95774-3d3f-4050-a13f-451a695419a0,8,3,NaN,1749642.0,63.0,0.0,63.0,2025-06-01 10:24:58
3,c93ed407-6af5-4816-a4d4-4be5626f23b7,10,3,NaN,691127.0,43.0,0.0,43.0,2025-06-01 10:25:00
4,78d9b721-67c7-426b-a963-ae25c9e3b4a5,5,1,NaN,412641.0,48.0,0.0,48.0,2025-06-01 10:25:03


### vi. `users`
Inspects the user/customer reference table to check available attributes and join keys (`user_id`).

In [21]:
inspect_subfolder("users", preview_rows=5)

Subfolder: `users` (24 CSV file(s))


,table,rows,cols,total_nulls,columns
0,users_202307,580,4,0,"user_id, gender, birthdate, registered_at"
1,users_202308,938,4,0,"user_id, gender, birthdate, registered_at"
2,users_202309,1601,4,0,"user_id, gender, birthdate, registered_at"
3,users_202310,2655,4,0,"user_id, gender, birthdate, registered_at"
4,users_202311,4340,4,0,"user_id, gender, birthdate, registered_at"
5,users_202312,7109,4,0,"user_id, gender, birthdate, registered_at"
6,users_202401,11576,4,0,"user_id, gender, birthdate, registered_at"
7,users_202402,17099,4,0,"user_id, gender, birthdate, registered_at"
8,users_202403,28320,4,0,"user_id, gender, birthdate, registered_at"
9,users_202404,40834,4,0,"user_id, gender, birthdate, registered_at"


Table: `users_202307` (580 rows)


,column,dtype,nulls,null_%,unique
0,user_id,int64,0,0.0,580
1,gender,str,0,0.0,2
2,birthdate,str,0,0.0,570
3,registered_at,str,0,0.0,580


,user_id,gender,birthdate,registered_at
0,1,female,1970-04-22,2023-07-01 08:13:07
1,2,female,1991-12-08,2023-07-01 09:53:48
2,3,female,1971-03-15,2023-07-01 10:07:00
3,4,male,1975-11-08,2023-07-01 10:28:08
4,5,male,2005-05-10,2023-07-01 11:02:21


Table: `users_202308` (938 rows)


... (64 outputs left out)

,user_id,gender,birthdate,registered_at
0,1833761,female,1998-10-04,2025-05-01 08:17:21
1,1833762,female,1986-12-07,2025-05-01 08:17:26
2,1833763,female,2009-02-15,2025-05-01 08:17:39
3,1833764,female,1967-06-02,2025-05-01 08:18:04
4,1833765,male,1966-04-20,2025-05-01 08:18:07


Table: `users_202506` (178,713 rows)


,column,dtype,nulls,null_%,unique
0,user_id,int64,0,0.0,178713
1,gender,str,0,0.0,2
2,birthdate,str,0,0.0,16435
3,registered_at,str,0,0.0,178713


,user_id,gender,birthdate,registered_at
0,2017545,male,2003-03-29,2025-06-01 07:01:18
1,2017546,female,1998-02-16,2025-06-01 07:01:27
2,2017547,male,2009-03-06,2025-06-01 07:01:28
3,2017548,female,1966-02-24,2025-06-01 07:02:18
4,2017549,male,1982-05-22,2025-06-01 07:02:31


### vii. `vouchers`
Inspects the promotional/discount voucher reference table to check voucher attributes and join keys (`voucher_id`).

In [22]:
inspect_subfolder("vouchers", preview_rows=5)

Subfolder: `vouchers` (1 CSV file(s))


,table,rows,cols,total_nulls,columns
0,vouchers,16,6,0,"voucher_id, voucher_code, discount_type, disco..."


Table: `vouchers` (16 rows)


,column,dtype,nulls,null_%,unique
0,voucher_id,int64,0,0.0,16
1,voucher_code,str,0,0.0,8
2,discount_type,str,0,0.0,2
3,discount_value,float64,0,0.0,8
4,valid_from,str,0,0.0,16
5,valid_to,str,0,0.0,16


,voucher_id,voucher_code,discount_type,discount_value,valid_from,valid_to
0,1,SALES77,percentage,7.0,2023-07-07,2023-07-09
1,2,SALES88,percentage,8.0,2023-08-08,2023-08-10
2,3,SALES99,percentage,9.0,2023-09-09,2023-09-11
3,4,SALES10,percentage,10.0,2023-10-10,2023-10-12
4,5,SALES11,percentage,11.0,2023-11-11,2023-11-13


### viii. Consolidated Summary for `transactions` and `transaction_items`
Merges all 24 monthly CSV partitions for `transactions` and `transaction_items`, checks schema consistency across partitions, verifies date coverage (`2023-07` to `2025-06`), and summarizes missing values and column schemas.

In [27]:
import pandas as pd

TARGET_GROUPS = {
    "transactions": "transactions_",
    "transaction_items": "transaction_items_",
}

all_csvs = sorted(DATA_DIR.rglob("*.csv"))
summary_records = []
merged_streams = {}

for group_label, prefix in TARGET_GROUPS.items():
  matched_files = [
      p for p in all_csvs if p.stem.lower().startswith(prefix.lower())
  ]
  assert matched_files, f"No CSV files matched prefix '{prefix}' under {DATA_DIR}"

  schemas = set()
  total_rows = 0
  null_counts = {}
  min_dt = None
  max_dt = None
  date_col = None
  dfs = []

  for p in matched_files:
    df = pd.read_csv(p, low_memory=False)
    schemas.add(tuple(df.columns))
    total_rows += len(df)
    dfs.append(df)

    # Accumulate nulls per column across partitions
    for col, cnt in df.isna().sum().items():
      null_counts[col] = null_counts.get(col, 0) + int(cnt)

    # Detect and track min/max timestamp across partitions
    if date_col is None:
      date_col = next(
          (
              c
              for c in df.columns
              if any(k in c.lower() for k in ("date", "created_at", "time"))
          ),
          None,
      )
    if date_col and date_col in df.columns and len(df) > 0:
      parsed_dt = pd.to_datetime(df[date_col], errors="coerce")
      file_min = parsed_dt.min()
      file_max = parsed_dt.max()
      if pd.notna(file_min) and (min_dt is None or file_min < min_dt):
        min_dt = file_min
      if pd.notna(file_max) and (max_dt is None or file_max > max_dt):
        max_dt = file_max

  merged_streams[group_label] = pd.concat(dfs, ignore_index=True)
  base_cols = list(dfs[0].columns)
  non_zero_nulls = {c: n for c, n in null_counts.items() if n > 0}

  summary_records.append({
      "stream": group_label,
      "files_merged": len(matched_files),
      "total_rows": total_rows,
      "cols": len(base_cols),
      "schema_consistent": len(schemas) == 1,
      "date_col": date_col,
      "min_timestamp": str(min_dt) if min_dt is not None else None,
      "max_timestamp": str(max_dt) if max_dt is not None else None,
      "columns_with_nulls": str(non_zero_nulls) if non_zero_nulls else "None",
  })

  print(f"[{group_label}] columns ({len(base_cols)}): {', '.join(base_cols)}")

merged_summary_df = pd.DataFrame(summary_records)
display(merged_summary_df)

[transactions] columns (9): transaction_id, store_id, payment_method_id, voucher_id, user_id, original_amount, discount_applied, final_amount, created_at
[transaction_items] columns (6): transaction_id, item_id, quantity, unit_price, subtotal, created_at


,stream,files_merged,total_rows,cols,schema_consistent,date_col,min_timestamp,max_timestamp,columns_with_nulls
0,transactions,24,14623691,9,True,created_at,2023-07-01 07:00:00,2025-06-30 19:59:39,"{'voucher_id': 14006140, 'user_id': 7310827}"
1,transaction_items,24,29246323,6,True,created_at,2023-07-01 07:00:00,2025-06-30 19:59:39,None


## C. Data Quality and Relational Integrity Audit

In [28]:
import pandas as pd

tx_df = merged_streams["transactions"]
ti_df = merged_streams["transaction_items"]

print("=== 1. Transactions Sanity & Arithmetic Checks ===")
print("Total rows:", f"{len(tx_df):,}")
print("Unique transaction_id:", f"{tx_df['transaction_id'].nunique():,}")
print("Non-positive original_amount:", int((tx_df["original_amount"] <= 0).sum()))
print("Negative discount_applied:", int((tx_df["discount_applied"] < 0).sum()))
print("Non-positive final_amount:", int((tx_df["final_amount"] <= 0).sum()))

amt_diff = (tx_df["original_amount"] - tx_df["discount_applied"] - tx_df["final_amount"]).abs()
print("Rows where |original - discount - final| > 0.01:", int((amt_diff > 0.01).sum()))

print("\n=== 2. Transaction Items Sanity & Arithmetic Checks ===")
print("Total rows:", f"{len(ti_df):,}")
print("Non-positive quantity:", int((ti_df["quantity"] <= 0).sum()))
print("Non-positive unit_price:", int((ti_df["unit_price"] <= 0).sum()))

line_diff = (ti_df["quantity"] * ti_df["unit_price"] - ti_df["subtotal"]).abs()
print("Rows where |qty * unit_price - subtotal| > 0.01:", int((line_diff > 0.01).sum()))

print("\n=== 3. Foreign Key & Orphan Checks ===")
tx_ids = set(tx_df["transaction_id"].dropna().unique())
ti_tx_ids = set(ti_df["transaction_id"].dropna().unique())
print("Items referencing missing transaction_id:", len(ti_tx_ids - tx_ids))
print("Transactions with zero line items:", len(tx_ids - ti_tx_ids))

if "menu_items" in loaded_tables:
  menu_ids = set(loaded_tables["menu_items"]["item_id"].dropna().unique())
  item_ids = set(ti_df["item_id"].dropna().unique())
  print("Items referencing missing item_id in menu_items:", len(item_ids - menu_ids))

if "stores" in loaded_tables:
  store_ids = set(loaded_tables["stores"]["store_id"].dropna().unique())
  tx_store_ids = set(tx_df["store_id"].dropna().unique())
  print("Transactions referencing missing store_id in stores:", len(tx_store_ids - store_ids))

if "payment_methods" in loaded_tables:
  pm_ids = set(loaded_tables["payment_methods"]["method_id"].dropna().unique())
  tx_pm_ids = set(tx_df["payment_method_id"].dropna().unique())
  print("Transactions referencing missing payment_method_id:", len(tx_pm_ids - pm_ids))

=== 1. Transactions Sanity & Arithmetic Checks ===
Total rows: 14,623,691
Unique transaction_id: 14,623,691
Non-positive original_amount: 0
Negative discount_applied: 0
Non-positive final_amount: 0
Rows where |original - discount - final| > 0.01: 0

=== 2. Transaction Items Sanity & Arithmetic Checks ===
Total rows: 29,246,323
Non-positive quantity: 0
Non-positive unit_price: 0
Rows where |qty * unit_price - subtotal| > 0.01: 0

=== 3. Foreign Key & Orphan Checks ===
Items referencing missing transaction_id: 0
Transactions with zero line items: 0
Items referencing missing item_id in menu_items: 0
Transactions referencing missing store_id in stores: 0
Transactions referencing missing payment_method_id: 0


# **III. Data Preprocessing**

## A. Register Merged Streams & Reference Tables in SQLite

In [29]:
import pandas as pd

# Register merged 24-month streams in SQLite
merged_streams["transactions"].to_sql(
    "transactions_full", conn, if_exists="replace", index=False
)
merged_streams["transaction_items"].to_sql(
    "transaction_items_full", conn, if_exists="replace", index=False
)

# Register remaining reference subfolders if not already in loaded_tables
for ref_name in ["users", "vouchers"]:
  ref_dir = DATA_DIR / ref_name
  if ref_dir.exists() and ref_name not in loaded_tables:
    ref_files = sorted(ref_dir.rglob("*.csv"))
    if ref_files:
      ref_df = pd.concat(
          [pd.read_csv(p, low_memory=False) for p in ref_files],
          ignore_index=True,
      )
      ref_df.to_sql(ref_name, conn, if_exists="replace", index=False)
      loaded_tables[ref_name] = ref_df

# Index key join columns for faster SQL joins
cur = conn.cursor()
cur.execute(
    "CREATE INDEX IF NOT EXISTS idx_tx_id ON transactions_full(transaction_id);"
)
cur.execute(
    "CREATE INDEX IF NOT EXISTS idx_ti_tx_id ON"
    " transaction_items_full(transaction_id);"
)
cur.execute(
    "CREATE INDEX IF NOT EXISTS idx_ti_item_id ON"
    " transaction_items_full(item_id);"
)
conn.commit()

db_objects = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view')"
    " ORDER BY name;",
    conn,
)
display(db_objects)

,name,type
0,menu_items,table
1,payment_methods,table
2,stores,table
3,transaction_items_202307,table
4,transaction_items_202308,table
...,...,...
74,users_202503,table
75,users_202504,table
76,users_202505,table
77,users_202506,table


Consolidates the 24 month CSV files into two persistent queryable SQLite tables, attaches the lookup tables, and builds database indexes. Joining almost 29.3 million line items against 14.6 million transactions directly in Pandas requires massive RAM and risks crashing Jupyter. Indexing the join keys in SQLite lets the database match rows efficiently in memory without copying giant DataFrames repeatedly. 

## B. Build the Integrated Analytical view

In [30]:
# Check optional dimension tables so the SQL query adapts safely
existing_names = set(db_objects["name"])
has_vouchers_tbl = "vouchers" in existing_names

# Inspect voucher key column name if vouchers table exists
voucher_join_sql = ""
voucher_select_sql = (
    "0 AS voucher_matched, CAST(NULL AS TEXT) AS voucher_label,"
)
if has_vouchers_tbl:
  v_cols = pd.read_sql_query("SELECT * FROM vouchers LIMIT 1;", conn).columns.tolist()
  v_key = next((c for c in v_cols if "voucher" in c.lower() and "id" in c.lower()), None)
  if v_key:
    voucher_join_sql = f"LEFT JOIN vouchers v ON t.voucher_id = v.{v_key}"
    voucher_select_sql = (
        f"CASE WHEN v.{v_key} IS NOT NULL THEN 1 ELSE 0 END AS"
        " voucher_matched,"
    )

create_view_sql = f"""
DROP VIEW IF EXISTS vw_order_items_enriched;
CREATE VIEW vw_order_items_enriched AS
SELECT
    ti.transaction_id,
    ti.item_id,
    m.item_name,
    m.category AS item_category,
    m.price AS catalog_price,
    m.is_seasonal,
    ti.quantity,
    ti.unit_price,
    ti.subtotal AS line_subtotal,
    t.store_id,
    s.store_name,
    s.city AS store_city,
    s.state AS store_state,
    t.payment_method_id,
    pm.method_name AS payment_method,
    pm.category AS payment_category,
    t.voucher_id,
    CASE WHEN t.voucher_id IS NOT NULL THEN 1 ELSE 0 END AS has_voucher,
    {voucher_select_sql}
    t.user_id,
    CASE WHEN t.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_registered_user,
    t.original_amount AS tx_original_amount,
    t.discount_applied AS tx_discount_applied,
    t.final_amount AS tx_final_amount,
    ti.created_at AS item_created_at,
    t.created_at AS tx_created_at,
    SUBSTR(ti.created_at, 1, 10) AS order_date,
    SUBSTR(ti.created_at, 1, 7) AS order_year_month,
    CAST(SUBSTR(ti.created_at, 12, 2) AS INTEGER) AS order_hour,
    CAST(STRFTIME('%w', ti.created_at) AS INTEGER) AS day_of_week_num,
    CASE CAST(STRFTIME('%w', ti.created_at) AS INTEGER)
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS day_of_week
FROM transaction_items_full ti
INNER JOIN transactions_full t
    ON ti.transaction_id = t.transaction_id
LEFT JOIN menu_items m
    ON ti.item_id = m.item_id
LEFT JOIN stores s
    ON t.store_id = s.store_id
LEFT JOIN payment_methods pm
    ON t.payment_method_id = pm.method_id
{voucher_join_sql};
"""

conn.executescript(create_view_sql)
conn.commit()

verification_df = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_joined_rows,
        COUNT(DISTINCT transaction_id) AS unique_transactions,
        COUNT(DISTINCT item_id) AS unique_items,
        MIN(order_date) AS min_date,
        MAX(order_date) AS max_date,
        SUM(has_voucher) AS rows_with_voucher,
        SUM(is_registered_user) AS rows_with_registered_user
    FROM vw_order_items_enriched;
    """,
    conn,
)
display(verification_df)
display(
    pd.read_sql_query("SELECT * FROM vw_order_items_enriched LIMIT 5;", conn)
)

,total_joined_rows,unique_transactions,unique_items,min_date,max_date,rows_with_voucher,rows_with_registered_user
0,29246323,14623691,8,2023-07-01,2025-06-30,1234158,14625050


,transaction_id,item_id,item_name,item_category,catalog_price,is_seasonal,quantity,unit_price,line_subtotal,store_id,...,tx_original_amount,tx_discount_applied,tx_final_amount,item_created_at,tx_created_at,order_date,order_year_month,order_hour,day_of_week_num,day_of_week
0,2ae6d188-76c2-4095-b861-ab97d3cd9312,6,Mocha,coffee,9.5,0,3,9.5,28.5,4,...,38.0,0.0,38.0,2023-07-01 07:00:00,2023-07-01 07:00:00,2023-07-01,2023-07,7,6,Saturday
1,2ae6d188-76c2-4095-b861-ab97d3cd9312,6,Mocha,coffee,9.5,0,1,9.5,9.5,4,...,38.0,0.0,38.0,2023-07-01 07:00:00,2023-07-01 07:00:00,2023-07-01,2023-07,7,6,Saturday
2,7d0a474d-62f4-442a-96b6-a5df2bda8832,5,Flat White,coffee,9.0,0,3,9.0,27.0,7,...,33.0,0.0,33.0,2023-07-01 07:00:02,2023-07-01 07:00:02,2023-07-01,2023-07,7,6,Saturday
3,7d0a474d-62f4-442a-96b6-a5df2bda8832,1,Espresso,coffee,6.0,0,1,6.0,6.0,7,...,33.0,0.0,33.0,2023-07-01 07:00:02,2023-07-01 07:00:02,2023-07-01,2023-07,7,6,Saturday
4,85f86fef-fddb-4eef-9dc3-1444553e6108,7,Hot Chocolate,non-coffee,9.0,0,3,9.0,27.0,1,...,27.0,0.0,27.0,2023-07-01 07:00:04,2023-07-01 07:00:04,2023-07-01,2023-07,7,6,Saturday


Creates a single unified virtual table that stitches together every line item with its transaction header, item details, store branch, payment channel, voucher / user flags, and parsed calendar attributes. Instead of writing multi-table JOINs and date-parsing logic every time you run an EDA query, Market Basket Analysis, or cost / inventory calculation, every downstream analysis simply queries. 

## C. Post-Join Verification & Scope Alignment Check

In [31]:
import pandas as pd

print("=== 1. Menu Items & Categories ===")
menu_summary_df = pd.read_sql_query(
    """
    SELECT
        item_id,
        item_name,
        item_category,
        catalog_price,
        is_seasonal,
        COUNT(*) AS line_item_rows,
        SUM(quantity) AS total_units_sold,
        ROUND(SUM(line_subtotal), 2) AS total_gross_subtotal
    FROM vw_order_items_enriched
    GROUP BY item_id, item_name, item_category, catalog_price, is_seasonal
    ORDER BY total_units_sold DESC;
    """,
    conn,
)
display(menu_summary_df)

print("\n=== 2. Duplicate (transaction_id, item_id) Line Items ===")
dup_lines_df = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS duplicate_tx_item_pairs,
        SUM(line_count - 1) AS extra_rows_from_split_lines
    FROM (
        SELECT
            transaction_id,
            item_id,
            COUNT(*) AS line_count
        FROM vw_order_items_enriched
        GROUP BY transaction_id, item_id
        HAVING COUNT(*) > 1
    );
    """,
    conn,
)
display(dup_lines_df)

print("\n=== 3. Store & Payment Channel Coverage ===")
store_summary_df = pd.read_sql_query(
    """
    SELECT
        store_id,
        store_name,
        store_state,
        COUNT(DISTINCT transaction_id) AS transactions_count,
        COUNT(*) AS line_items_count,
        ROUND(SUM(tx_final_amount), 2) AS sum_tx_final_amount
    FROM (
        SELECT DISTINCT
            transaction_id,
            store_id,
            store_name,
            store_state,
            tx_final_amount
        FROM vw_order_items_enriched
    )
    GROUP BY store_id, store_name, store_state
    ORDER BY transactions_count DESC;
    """,
    conn,
)
display(store_summary_df)

=== 1. Menu Items & Categories ===


,item_id,item_name,item_category,catalog_price,is_seasonal,line_item_rows,total_units_sold,total_gross_subtotal
0,2,Americano,coffee,7.0,0,3659903,7320638,51244466.0
1,4,Cappuccino,coffee,8.0,0,3657029,7314603,58516824.0
2,7,Hot Chocolate,non-coffee,9.0,0,3656182,7313522,65821698.0
3,3,Latte,coffee,8.0,0,3656267,7311169,58489352.0
4,1,Espresso,coffee,6.0,0,3656538,7310463,43862778.0
5,5,Flat White,coffee,9.0,0,3653934,7307693,65769237.0
6,6,Mocha,coffee,9.5,0,3654374,7307251,69418884.5
7,8,Matcha Latte,non-coffee,10.0,0,3652096,7303687,73036870.0



=== 2. Duplicate (transaction_id, item_id) Line Items ===


,duplicate_tx_item_pairs,extra_rows_from_split_lines
0,2284725,2360635



=== 3. Store & Payment Channel Coverage ===


,store_id,store_name,store_state,transactions_count,line_items_count,sum_tx_final_amount
0,5,G Coffee @ Seksyen 21,Putrajaya,1464715,1464715,48436103.56
1,6,G Coffee @ Alam Tun Hussein Onn,Selangor Darul Ehsan,1463334,1463334,48382642.08
2,8,G Coffee @ Bandar Seri Mulia,Kuala Lumpur,1463207,1463207,48436163.92
3,10,G Coffee @ Taman Damansara,Selangor Darul Ehsan,1463106,1463106,48424478.83
4,4,G Coffee @ Kampung Changkat,Putrajaya,1462887,1462887,48403142.65
5,3,G Coffee @ USJ 57W,Putrajaya,1461980,1461980,48384445.45
6,7,G Coffee @ Damansara Saujana,Selangor Darul Ehsan,1461538,1461538,48336540.34
7,2,G Coffee @ Kondominium Putra,Selangor Darul Ehsan,1461153,1461153,48368169.38
8,9,G Coffee @ PJS8,Putrajaya,1460997,1460997,48339997.92
9,1,G Coffee @ USJ 89q,Kuala Lumpur,1460774,1460774,48333710.30


## Data Understanding and Verification Report: Menu Catalog and Line-Item Structure

**Overview** 


A post-join verification pass across the integrated coffee shop dataset evaluated menu catalog composition, unit and subtotal distributions, and line-item granularity within individual orders. These findings establish the baseline scope and aggregation rules for downstream exploratory analysis, market basket analysis, and cost-aware demand and inventory modeling. 


**1. Menu Items and Categories**

- The catalog contains 8 unique menu items split across 6 coffee items (Americano, Cappuccino, Latte, Espresso, Flat White, and Mocha) and 2 non-coffee items (Hot Chocolate and Matcha Latte). All 8 items have is_seasonal = 0, indicating year-round availability rather than seasonal menu rotations.

- Line-item representation and unit sales are evenly distributed across the catalog, with each item accounting for roughly 3.65 million line-item rows and approximately 7.3 million units sold, combining for roughly 58.5 million total units across all 8 items.

- Because unit volumes are balanced across items, gross line subtotals scale directly with item price, from lower priced staples such as Espresso (6.0) and Americano (7.0) up to higher priced items such as Mocha (9.5) and Matcha Latte (10.0).


**2. Split or Duplicate (transaction_id, item_id) Lines** 

- Multi Line Item Entries: Approximately 2.2 million (2,284,725 exact) (transaction_id, item_id) pairs appear more than once within the same transaction (2,360,635 additional line item rows), reflecting orders where the same item is recorded across multiple separate lines on a single receipt.

**Implications for Downstream Analysis:**

- **Volume and Revenue Aggregation:** Summing quantity and line_subtotal across all line items preserves accurate total units and gross sales, whereas transaction level totals (tx_original_amount, tx_discount_applied, tx_final_amount) must be deduplicated by transaction_id before summing to prevent double counting.

- **Market Basket Analysis:** Baskets must be grouped by transaction_id and deduplicated by (transaction_id, item_id) or item_name so repeated line entries of the same item within one order do not artificially inflate item co occurrence counts.

## Additional Process

This step was added in order to speed up the queries since it takes about 4-7 minutes when I run each cell

**1. Materialize order_items_enriched & Index Key Columns**

In [32]:
import pandas as pd

conn.executescript("""
DROP TABLE IF EXISTS order_items_enriched;
CREATE TABLE order_items_enriched AS
SELECT * FROM vw_order_items_enriched;

CREATE INDEX IF NOT EXISTS idx_oie_tx_id ON order_items_enriched(transaction_id);
CREATE INDEX IF NOT EXISTS idx_oie_date ON order_items_enriched(order_date);
CREATE INDEX IF NOT EXISTS idx_oie_ym ON order_items_enriched(order_year_month);
CREATE INDEX IF NOT EXISTS idx_oie_item ON order_items_enriched(item_id);
""")
conn.commit()

print("Materialized row count:", pd.read_sql_query("SELECT COUNT(*) AS n FROM order_items_enriched;", conn).iloc[0, 0])

Materialized row count: 29246323


**2. Core EDA Aggregation**

A. Monthly Trend (2023-07 to 2025-06)

In [33]:
monthly_eda_df = pd.read_sql_query(
    """
    WITH tx_clean AS (
        SELECT DISTINCT
            transaction_id,
            order_year_month,
            has_voucher,
            is_registered_user,
            tx_original_amount,
            tx_discount_applied,
            tx_final_amount
        FROM order_items_enriched
    ),
    ti_clean AS (
        SELECT
            transaction_id,
            SUM(quantity) AS total_units,
            SUM(line_subtotal) AS gross_item_subtotal
        FROM order_items_enriched
        GROUP BY transaction_id
    )
    SELECT
        t.order_year_month,
        COUNT(*) AS total_transactions,
        SUM(i.total_units) AS total_units_sold,
        ROUND(SUM(i.gross_item_subtotal), 2) AS gross_item_subtotal,
        ROUND(SUM(t.tx_discount_applied), 2) AS total_discounts,
        ROUND(SUM(t.tx_final_amount), 2) AS net_final_amount,
        ROUND(AVG(t.tx_final_amount), 2) AS avg_order_value,
        ROUND(100.0 * AVG(t.has_voucher), 2) AS pct_with_voucher,
        ROUND(100.0 * AVG(t.is_registered_user), 2) AS pct_registered_user
    FROM tx_clean t
    JOIN ti_clean i ON t.transaction_id = i.transaction_id
    GROUP BY t.order_year_month
    ORDER BY t.order_year_month;
    """,
    conn,
)
display(monthly_eda_df)

,order_year_month,total_transactions,total_units_sold,gross_item_subtotal,total_discounts,net_final_amount,avg_order_value,pct_with_voucher,pct_registered_user
0,2023-07,620262,2483420,20641393.0,98192.29,20543200.71,33.12,6.81,0.31
1,2023-08,619980,2478589,20599129.0,181070.64,20418058.36,32.93,9.01,0.53
2,2023-09,600643,2400629,19950715.0,265037.04,19685677.96,32.77,11.64,0.87
3,2023-10,619396,2477223,20590651.0,139617.90,20451033.10,33.02,6.78,1.43
4,2023-11,600545,2403455,19980600.5,154400.29,19826200.21,33.01,7.03,2.37
5,2023-12,620777,2479440,20609261.5,0.00,20609261.50,33.20,0.00,3.79
6,2024-01,620358,2480908,20624421.0,0.00,20624421.00,33.25,0.00,6.21
7,2024-02,580012,2320593,19287423.5,0.00,19287423.50,33.25,0.00,9.80
8,2024-03,620481,2479749,20608848.0,0.00,20608848.00,33.21,0.00,15.11
9,2024-04,599847,2401293,19959788.5,0.00,19959788.50,33.27,0.00,22.73


B. Hourly & Day-of-Week Demand Pattern

In [34]:
hourly_eda_df = pd.read_sql_query(
    """
    WITH tx_clean AS (
        SELECT DISTINCT
            transaction_id,
            order_hour,
            day_of_week_num,
            day_of_week,
            tx_final_amount
        FROM order_items_enriched
    )
    SELECT
        order_hour,
        COUNT(*) AS total_transactions,
        ROUND(SUM(tx_final_amount), 2) AS net_final_amount,
        ROUND(AVG(tx_final_amount), 2) AS avg_order_value
    FROM tx_clean
    GROUP BY order_hour
    ORDER BY order_hour;
    """,
    conn,
)
display(hourly_eda_df)

,order_hour,total_transactions,net_final_amount,avg_order_value
0,7,319992,10592483.63,33.10
1,8,657891,21742488.43,33.05
2,9,1151967,38140484.30,33.11
3,10,1720424,56911890.51,33.08
4,11,2187446,72367456.95,33.08
5,12,2368659,78359504.38,33.08
6,13,2185617,72296878.70,33.08
7,14,1720672,56962770.58,33.10
8,15,1152642,38143431.19,33.09
9,16,658904,21792919.77,33.07


**Exploratory Data Analysis (EDA) Summary: Monthly and Hourly Demand Patterns**


A two year exploratory analysis (July 2023 to June 2025, covering 14,623,691 transactions and 29,246,323 line items) across the materialized order_items_enriched table highlights three core operational patterns:

- Calendar Driven Monthly Volume and Stable Spend: Monthly order volume scales closely with calendar days, averaging roughly ~620k transactions / ~2.48M units in 31 day months, ~600k transactions / ~2.40M units in 30 day months, and 580,012 (February 2024, 29 days) versus 560,492 (February 2025, 28 days) in February, while average order value remains tightly consistent between $32.77 and $33.28.

- Cyclical Voucher Campaigns and Rising Registered User Share: Voucher usage and discounts follow a consistent annual cycle, with zero discounts from December through April in both years, ramping up in May (~2.27%), peaking in September (~11.65% of transactions; ~$265k in discounts), and tapering through November, while registered user attribution rises steadily from 0.31% in July 2023 to ~99.0% by May and June 2025.

- Symmetric Midday Traffic Peak: Hourly transaction volume rises from 07:00 (319,992 orders) to a distinct midday peak at 12:00 (2,368,659 orders; $78.36M net sales) before declining symmetrically through 19:00 (46,721 orders). Because average order value stays near $32.94 to $33.17 across operating hours, revenue surges are driven primarily by order volume rather than larger basket sizes.


Modeling Note: Because registered user share trends strongly upward over time and discount campaigns follow a seasonal calendar, downstream classification or demand models should use a time based train/test split rather than random sampling to avoid temporal leakage.